In [ ]:
from pathlib import Path
from datetime import datetime

import requests

In [ ]:
BPA_DATA_DIR = Path("data/bpa")

BASE_URL = (
    "https://transmission.bpa.gov/Business/Operations/Outages/" "OutagesCY{year}.xlsx"
)

START_YEAR = 2008
END_YEAR = datetime.now().year

In [ ]:
def download_bpa_outages(
    output_dir: Path = BPA_DATA_DIR,
    start_year: int = START_YEAR,
    end_year: int = END_YEAR,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": (
                "Mozilla/5.0 BPA outage dataset downloader " "(academic research)"
            )
        }
    )

    for year in range(start_year, end_year + 1):
        url = BASE_URL.format(year=year)
        output_path = output_dir / f"OutagesCY{year}.xlsx"

        if output_path.exists():
            print(f"[SKIP] {year}: already exists")
            continue

        print(f"[DOWNLOAD] {year}: {url}")

        try:
            response = session.get(url, timeout=60)
            response.raise_for_status()

            # Check that BPA actually returned an Excel file rather than
            # an HTML error page.
            content_type = response.headers.get("Content-Type", "").lower()

            if (
                "spreadsheet" not in content_type
                and "excel" not in content_type
                and not response.content.startswith(b"PK")
            ):
                print(
                    f"[WARNING] {year}: unexpected content type "
                    f"{content_type!r}; skipping"
                )
                continue

            output_path.write_bytes(response.content)

            size_mb = output_path.stat().st_size / 1024**2
            print(f"[OK] {year}: {output_path} ({size_mb:.2f} MB)")

        except requests.RequestException as exc:
            print(f"[FAILED] {year}: {exc}")

In [4]:
download_bpa_outages()

[DOWNLOAD] 2008: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2008.xlsx
[FAILED] 2008: 404 Client Error: Not Found for url: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2008.xlsx
[DOWNLOAD] 2009: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2009.xlsx
[FAILED] 2009: 404 Client Error: Not Found for url: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2009.xlsx
[DOWNLOAD] 2010: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2010.xlsx
[FAILED] 2010: 404 Client Error: Not Found for url: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2010.xlsx
[DOWNLOAD] 2011: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2011.xlsx
[FAILED] 2011: 404 Client Error: Not Found for url: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2011.xlsx
[DOWNLOAD] 2012: https://transmission.bpa.gov/Business/Operations/Outages/OutagesCY2012.xlsx
[FAILED] 2012: 404 Clie